In [ ]:
from google import genai

from google.genai import types


# Initialize your client

client = genai.Client(api_key="YOUR-API-KEY-HERE")


# --- 1. THE MOCK DATABASE ---

# This simulates an e-commerce backend

product_database = {

    "SKU-101": {"name": "Black Hoodie", "price": 45.00, "stock": 12},

    "SKU-102": {"name": "Coffee Mug", "price": 15.00, "stock": 19},

}


order_database = {

    "ORD-888": {"status": "Shipped", "item": "SKU-101", "customer_email": "test@example.com"},

    "ORD-999": {"status": "Processing", "item": "SKU-102", "customer_email": "hello@world.com"}

}


# --- 2. THE TOOLS (Your Agent's Hands) ---


def get_product_info(product_name: str) -> str:

    """Searches the catalog for a product and returns price and stock levels."""

    print(f"\n⚙️ SYSTEM: Searching catalog for '{product_name}'...")



    # Simple search logic

    for sku, details in product_database.items():

        if product_name.lower() in details["name"].lower():

            if details["stock"] > 0:

                return f"Found {details['name']}. Price is ${details['price']}. We have {details['stock']} in stock."

            else:

                return f"Found {details['name']}, but it is currently out of stock."



    return "Product not found in our catalog."


def check_order_status(order_id: str) -> str:

    """Checks the shipping status of a specific order ID."""

    print(f"\n⚙️ SYSTEM: Looking up order {order_id}...")



    order = order_database.get(order_id)

    if order:

        return f"Order {order_id} is currently: {order['status']}."

    return "Order ID not found. Please verify the number."


# --- 3. THE AGENT SETUP ---


def run_ecommerce_agent(user_message):

    print(f"\n👤 CUSTOMER: {user_message}")



    # We give the agent specific instructions on how to behave

    system_instruction = """

    You are a helpful customer service agent for an e-commerce store.

    Always be polite. Use your tools to look up information before answering.

    If an item is out of stock, apologize and suggest they check back later.

    """



    chat = client.chats.create(

        model='gemini-3.5-flash',

        config=types.GenerateContentConfig(

            system_instruction=system_instruction,

            tools=[get_product_info, check_order_status],

            temperature=0.2 # Keeps the agent focused and less "creative" with facts

        )

    )



    response = chat.send_message(user_message)

    print(f"🤖 AGENT: {response.text}")


# --- LET'S TEST THE LOGIC ---

print("--- TEST 1: Product Inquiry ---")

run_ecommerce_agent("Do you have any black hoodies in stock? And how much are they?")

print("\n--- TEST 2: Order Status ---")

run_ecommerce_agent("Where is my stuff? My order number is ORD-999.")

print("\n--- TEST 3: Asking Price ---")

run_ecommerce_agent("Hello, I would like to inquire how much 5 Coffee Mugs cost to buy?")

print("\n--- TEST 4: Asking Stocks ---")

run_ecommerce_agent("Greetings, How many Black hoodies and Coffee Mugs are there?")


--- TEST 1: Product Inquiry ---

👤 CUSTOMER: Do you have any black hoodies in stock? And how much are they?

⚙️ SYSTEM: Searching catalog for 'black hoodie'...
🤖 AGENT: Yes, we do! We have the Black Hoodie in stock. It is priced at $45.00, and we currently have 12 available.

--- TEST 2: Order Status ---

👤 CUSTOMER: Where is my stuff? My order number is ORD-999.

⚙️ SYSTEM: Looking up order ORD-999...
🤖 AGENT: Your order ORD-999 is currently processing. We will update you as soon as it ships!

--- TEST 3: Asking Price ---

👤 CUSTOMER: Hello, I would like to inquire how much 5 Coffee Mugs cost to buy?

⚙️ SYSTEM: Searching catalog for 'Coffee Mug'...
🤖 AGENT: Hello! 

A single Coffee Mug costs $15.00. Therefore, 5 Coffee Mugs would cost a total of $75.00. 

We currently have 19 in stock, so they are available to order! Let me know if you would like help with anything else.

--- TEST 4: Asking Stocks ---

👤 CUSTOMER: Greetings, How many Black hoodies and Coffee Mugs are there?

⚙️ SYSTE